# Final consolidated visual results

## tl;dr

1. Five of 88 verified weekly returns set a new within-function incumbent (5.7%).
2. Seven of eight GPs beat a historical-mean RMSE baseline, but calibration varies and F5–F7 under-cover at 95%.
3. Week 12 recommendation stability is strong for F4/F5 and materially weaker for several other functions; robustness is not realised performance.
4. The immutable ledger reconstructs the required post-Week-11 counts and keeps unreturned Week 12 proposals outside the observed dataset.

This notebook is the final reader-facing visual synthesis. Detailed methods remain in the weekly and GP-evaluation notebooks.

## Table of contents

1. [Overview](#overview)
2. [Objectives](#objectives)
3. [Evidence provenance](#evidence-provenance)
4. [Environment and setup](#environment-setup)
5. [Data validation](#data-validation)
6. [Descriptive EDA](#descriptive-eda)
7. [Visual EDA](#visual-eda)
8. [GP model configuration](#gp-model-configuration)
9. [GP-UCB values](#gp-ucb-values)
10. [Reproducibility checks](#reproducibility-checks)
11. [Conclusions and next steps](#conclusions-next-steps)


<a id="overview"></a>
## 1. Overview

Consolidated reader-facing visual summary using the executed evidence already present on main.

<a id="objectives"></a>
## 2. Objectives

Present the preserved cross-week results, diagnostics, robustness evidence, and status views.

<a id="evidence-provenance"></a>
## 3. Evidence provenance

## Context & Methods

The notebook reads only frozen repository artifacts: the returned-pair ledger, performance summary, rolling GP metrics, sensitivity analysis and Week 12 proposal ledger. It does not refit models or create new black-box evidence.

### Key assumptions

- Improvements are evaluated within each function because objective scales differ.
- Rolling folds are chronological but arise from adaptive queries, not an independent test set.
- Sensitivity rows are diagnostic experiments and were not submitted.
- Missing Week 12 returns remain missing; recommendations are not scored as realised outcomes.

<a id="environment-setup"></a>
## 4. Environment and setup

Load the existing result tables, figures, and plotting dependencies.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'Results' / 'query_output_ledger.csv').is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError('Repository root not found')

ledger = pd.read_csv(ROOT / 'Results' / 'query_output_ledger.csv')
performance = pd.read_csv(ROOT / 'Results' / 'performance_summary_weeks_01_to_13.csv')
metrics = pd.read_csv(ROOT / 'Results' / 'gp_validation_metrics.csv')
sensitivity = pd.read_csv(ROOT / 'Results' / 'week12_sensitivity_analysis.csv')
proposals = pd.read_csv(ROOT / 'Results' / 'bbo_query_ledger.csv')
print(f'Repository: {ROOT}')

Repository: /Users/john-peteramewu/Documents/Codex/2026-08-07/understand-the-project-explain-the-structure/work/publish-week11.wwIJ2I


<a id="data-validation"></a>
## 5. Data validation

## Data

### 1. Validate frozen evidence

In [2]:
EXPECTED_COUNTS = [21, 21, 26, 41, 31, 31, 41, 51]
assert len(ledger) == 88
assert ledger.groupby('function').size().eq(11).all()
assert len(metrics) == len(proposals) == 8
assert len(sensitivity) == 80
assert proposals['status'].eq('proposal_only_return_unavailable').all()
assert proposals['observation_count'].tolist() == EXPECTED_COUNTS
print('Validated: 88 returned pairs, 80 sensitivity rows, 8 unreturned proposals.')

Validated: 88 returned pairs, 80 sensitivity rows, 8 unreturned proposals.


<a id="descriptive-eda"></a>
## 6. Descriptive EDA

Summarise the preserved performance and evidence-status tables used by the final visual synthesis.

<a id="visual-eda"></a>
## 7. Visual EDA

## Results

### 2. Four-panel consolidated evidence

In [3]:
confirmed = performance[performance['Evidence status'].eq('Confirmed return')].copy()
improvements = confirmed.assign(improved=confirmed['Improvement'].fillna(0).gt(0)).groupby('Function')['improved'].sum()
improvements = improvements.reindex([f'F{i}' for i in range(1, 9)], fill_value=0)
distinct_recommendations = sensitivity.groupby('function')['submission_query'].nunique().reindex(range(1, 9))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colours = ['#4472C4' if value == 0 else '#D97706' for value in improvements]
axes[0, 0].bar(improvements.index, improvements.values, color=colours)
axes[0, 0].set(title='A. Verified incumbent improvements', xlabel='Function', ylabel='Improving returns (of 11)', ylim=(0, max(2, improvements.max() + 0.5)))

skill = metrics.set_index('function')['rmse_skill_vs_naive'].reindex(range(1, 9))
axes[0, 1].bar([f'F{i}' for i in skill.index], skill.values, color=np.where(skill >= 0, '#2E8B57', '#C44536'))
axes[0, 1].axhline(0, color='black', linewidth=0.8)
axes[0, 1].set(title='B. Rolling GP skill versus mean baseline', xlabel='Function', ylabel='RMSE skill (higher is better)')

coverage = metrics.set_index('function')[['coverage_50', 'coverage_80', 'coverage_95']].reindex(range(1, 9))
for column, colour, marker in [('coverage_50', '#4472C4', 'o'), ('coverage_80', '#D97706', 's'), ('coverage_95', '#7A284E', '^')]:
    axes[1, 0].plot([f'F{i}' for i in coverage.index], coverage[column], marker=marker, color=colour, label=column.replace('coverage_', '') + '% observed')
for nominal, colour in [(0.5, '#4472C4'), (0.8, '#D97706'), (0.95, '#7A284E')]:
    axes[1, 0].axhline(nominal, color=colour, alpha=0.25, linestyle='--')
axes[1, 0].set(title='C. Predictive-interval calibration', xlabel='Function', ylabel='Observed coverage', ylim=(0, 1.05))
axes[1, 0].legend(fontsize=8)

axes[1, 1].bar([f'F{i}' for i in distinct_recommendations.index], distinct_recommendations.values, color='#6A5ACD')
axes[1, 1].set(title='D. Recommendation sensitivity', xlabel='Function', ylabel='Distinct queries across 10 settings', ylim=(0, 10))

fig.suptitle('Bayesian optimisation capstone — final consolidated results', fontsize=16, fontweight='bold')
fig.tight_layout(rect=(0, 0, 1, 0.96))
figure_path = ROOT / 'Figures' / 'final_consolidated_results.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
display(fig)
plt.close(fig)
print(figure_path.relative_to(ROOT))

<Figure size 1400x1000 with 4 Axes>

Figures/final_consolidated_results.png


### 3. Interpretation

- **Panel A:** only five historical returns improved an incumbent; sparse improvements do not mean the other observations were useless.
- **Panel B:** F2 is the only GP below the historical-mean RMSE baseline.
- **Panel C:** F5–F7 materially under-cover at 95%, while F1/F8 are conservative. Eleven folds per function make coverage estimates coarse.
- **Panel D:** one distinct query means stability across acquisition and bound settings (F4/F5/F7 in this appendix); many distinct queries indicate model or acquisition sensitivity, not necessarily poor realised performance.


<a id="gp-model-configuration"></a>
## 8. GP model configuration

Model configuration is inherited from the canonical weekly analyses and is not changed in this consolidated visual notebook.

<a id="gp-ucb-values"></a>
## 9. GP-UCB values

GP-UCB values are presented through the preserved weekly proposal and robustness evidence already loaded above.

<a id="reproducibility-checks"></a>
## 10. Reproducibility checks

### 4. Compact evidence table

In [4]:
summary = pd.DataFrame({
    'function': [f'F{i}' for i in range(1, 9)],
    'verified_observations': proposals['observation_count'].to_numpy(),
    'improving_returns': improvements.to_numpy(dtype=int),
    'rmse_skill_vs_mean': skill.to_numpy(),
    'coverage_95': coverage['coverage_95'].to_numpy(),
    'distinct_sensitivity_queries': distinct_recommendations.to_numpy(dtype=int),
    'week12_return_available': False,
})
display(summary.round({'rmse_skill_vs_mean': 3, 'coverage_95': 3}))

,function,verified_observations,improving_returns,rmse_skill_vs_mean,coverage_95,distinct_sensitivity_queries,week12_return_available
0,F1,21,0,0.317,1.000,9,False
1,F2,21,0,-0.095,1.000,8,False
2,F3,26,0,0.295,0.909,7,False
3,F4,41,1,0.712,0.909,1,False
4,F5,31,1,0.438,0.727,1,False
5,F6,31,0,0.512,0.727,3,False
6,F7,41,1,0.495,0.636,1,False
7,F8,51,2,0.897,1.000,3,False


<a id="conclusions-next-steps"></a>
## 11. Conclusions and next steps

## Takeaways

The project’s strongest conclusion is methodological rather than a claim of global optimality: progressively stronger Bayesian optimisation was paired with increasingly strict evidence controls. The final state distinguishes observed improvement, surrogate accuracy and calibration, recommendation robustness, and data lineage. Low-kappa Week 12 proposals are reproducible and portal-valid, but remain proposals until authoritative returns exist. The next scientifically valuable step is to obtain those returns, append them immutably, and evaluate realised improvement against the pre-registered recommendations.